In [ ]:
import os
import json
import httpx
import asyncpg
import asyncio
from typing import Annotated, TypedDict, List, Union
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

# --- CONFIGURATION ---
SESSION_FARMER_ID = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb" # Hard-coded for this session
MANDI_JSON_PATH = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_URL = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

# --- TOOL DEFINITION ---
@tool
async def get_market_price(mandi_name: str = "", crop: str = "") -> str:
    """
    Fetches live mandi prices or enlists available mandis.
    If mandi_name is empty, it returns a list of available mandis in the farmer's district.
    If mandi_name is provided, it fetches live price data from the API.
    """
    # 1. Fetch Farmer Context from DB
    conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
    row = await conn.fetchrow("SELECT state_name, dist_name FROM farmers WHERE id = $1", SESSION_FARMER_ID)
    await conn.close()
    
    if not row:
        return f"Error: Farmer ID {SESSION_FARMER_ID} not found in database."
    
    state, district = row["state_name"], row["dist_name"]

    # 2. Case: User hasn't specified a mandi -> Enlist from JSON
    if not mandi_name:
        if not os.path.exists(MANDI_JSON_PATH):
            return "Error: Mandi master data file not found."
            
        with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
            mandi_data = json.load(f)
            
        # Find mandis for this state/district (case-insensitive)
        available_mandis = []
        for s_key, districts in mandi_data.items():
            if s_key.lower() == state.lower():
                for d_key, mandis in districts.items():
                    if d_key.lower() == district.lower():
                        available_mandis = mandis
                        break
        
        if not available_mandis:
            return f"I found your location as {district}, {state}, but I couldn't find any specific mandis listed in my master data for this area."
        
        return f"The available mandis in {district} are: {', '.join(available_mandis)}. Which one would you like to check?"

    # 3. Case: User specified a mandi -> Call Live API
    clean_mandi = mandi_name.replace(" APMC", "").strip()
    params = {
        "api-key": os.getenv("AGMARKNET_API_KEY"),
        "format": "json",
        "filters[state]": state,
        "filters[market]": clean_mandi,
        "filters[commodity]": crop or "Wheat" # Default to Wheat if not specified
    }

    async with httpx.AsyncClient() as client:
        resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    
    if resp.status_code != 200:
        return f"API Error: Could not connect to Agmarknet (Status {resp.status_code})."

    data = resp.json()
    records = data.get("records", [])
    if not records:
        return f"No live price records found for {crop} at {mandi_name} currently."

    rec = records[0]
    return f"Success! The price for {rec['commodity']} at {rec['market']} on {rec['arrival_date']} is ₹{rec['modal_price']} per Quintal."


# --- LANGRAPH SETUP ---
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], "The conversation history"]

# Define the Model
model = ChatGroq(
            model=groq_model,
            temperature=0.2,
            api_key=os.getenv("GROQ_API_KEY"))

# Node: Agent Logic
async def call_model(state: AgentState):
    messages = state["messages"]
    response = await model.ainvoke(messages)
    return {"messages": [response]}

# Node: Tool Execution
async def call_tool(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    
    tool_results = []
    for tool_call in last_message.tool_calls:
        # Manually calling our async tool
        result = await get_market_price.ainvoke(tool_call["args"])
        tool_results.append(ToolMessage(
            tool_call_id=tool_call["id"],
            content=str(result)
        ))
    return {"messages": tool_results}

# Define Router
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# Build Graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", call_tool)
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

app = workflow.compile()

# --- INTERACTIVE CHAT FUNCTION ---
async def chat_with_farmer():
    print(f"🤖 KisanSaathi is ready! (Farmer ID: {SESSION_FARMER_ID})")
    print("Type 'quit' to exit.\n")
    
    state = {"messages": [
        AIMessage(content="Namaste! Main aapka KisanSaathi hoon. Aapke area ke mandi prices check karne mein main aapki madad kar sakta hoon. Aap kis crop ka bhav janna chahte hain?")
    ]}
    
    print(f"AI: {state['messages'][0].content}")

    while True:
        user_input = input("Farmer: ")
        if user_input.lower() in ["quit", "exit", "bye"]:
            break
            
        state["messages"].append(HumanMessage(content=user_input))
        
        # Stream the graph execution
        async for output in app.astream(state):
            for node, values in output.items():
                last_msg = values["messages"][-1]
                
                if node == "agent" and not last_msg.tool_calls:
                    print(f"AI: {last_msg.content}")
                elif node == "tools":
                    print(f"🛠️ [Tool Call] fetching data...")
                
                # Update local state history
                state["messages"].append(last_msg)

# To start the chat, run:
# await chat_with_farmer()


In [2]:
await chat_with_farmer()

🤖 KisanSaathi is ready! (Farmer ID: c59f6f44-1a98-4eaa-8cf0-3581316a32bb)
Type 'quit' to exit.

AI: Namaste! Main aapka KisanSaathi hoon. Aapke area ke mandi prices check karne mein main aapki madad kar sakta hoon. Aap kis crop ka bhav janna chahte hain?


ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [3]:
import os
import json
import httpx
import asyncpg
import asyncio
from typing import Annotated, TypedDict, List, Union
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq  # Updated import
from dotenv import load_dotenv

load_dotenv()

# --- CONFIGURATION ---
SESSION_FARMER_ID = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
MANDI_JSON_PATH = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_URL = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

# --- TOOL DEFINITION ---
@tool
async def get_market_price(mandi_name: str = "", crop: str = "") -> str:
    """
    Fetches live mandi prices or enlists available mandis.
    If mandi_name is empty, it returns a list of available mandis in the farmer's district.
    If mandi_name is provided, it fetches live price data from the API.
    """
    conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
    row = await conn.fetchrow("SELECT state_name, dist_name FROM farmers WHERE id = $1", SESSION_FARMER_ID)
    await conn.close()
    
    if not row:
        return f"Error: Farmer ID {SESSION_FARMER_ID} not found in database."
    
    state, district = row["state_name"], row["dist_name"]

    if not mandi_name:
        if not os.path.exists(MANDI_JSON_PATH):
            return "Error: Mandi master data file not found."
        with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
            mandi_data = json.load(f)
        
        available_mandis = []
        for s_key, districts in mandi_data.items():
            if s_key.lower() == state.lower():
                for d_key, mandis in districts.items():
                    if d_key.lower() == district.lower():
                        available_mandis = mandis
                        break
        
        if not available_mandis:
            return f"Location: {district}, {state}. No specific mandis found in master data."
        
        return f"Available mandis in {district}: {', '.join(available_mandis)}. Which one should I check?"

    # Call Live API
    clean_mandi = mandi_name.replace(" APMC", "").strip()
    params = {
        "api-key": os.getenv("AGMARKNET_API_KEY"),
        "format": "json",
        "filters[state]": state,
        "filters[market]": clean_mandi,
        "filters[commodity]": crop or "Wheat"
    }

    async with httpx.AsyncClient() as client:
        resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    
    if resp.status_code != 200:
        return f"API Error: Status {resp.status_code}."

    data = resp.json()
    records = data.get("records", [])
    if not records:
        return f"No price records found for {crop} at {mandi_name} currently."

    rec = records[0]
    return f"Success! Price for {rec['commodity']} at {rec['market']} on {rec['arrival_date']} is ₹{rec['modal_price']} per Quintal."


# --- LANGRAPH SETUP ---
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], "The conversation history"]

# Define the Groq Model
# Note: Using llama-3.3-70b-versatile as it has excellent tool-calling support
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
).bind_tools([get_market_price])

# Node: Agent Logic
async def call_model(state: AgentState):
    messages = state["messages"]
    response = await model.ainvoke(messages)
    return {"messages": [response]}

# Node: Tool Execution
async def call_tool(state: AgentState):
    messages = state["messages"]
    last_message = messages[-1]
    
    tool_results = []
    for tool_call in last_message.tool_calls:
        result = await get_market_price.ainvoke(tool_call["args"])
        tool_results.append(ToolMessage(
            tool_call_id=tool_call["id"],
            content=str(result)
        ))
    return {"messages": tool_results}

# Define Router
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# Build Graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", call_tool)
workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

app = workflow.compile()

# --- INTERACTIVE CHAT FUNCTION ---
async def chat_with_farmer():
    print(f"🤖 KisanSaathi (Groq Edition) is ready! (ID: {SESSION_FARMER_ID})")
    print("Type 'quit' to exit.\n")
    
    state = {"messages": [
        AIMessage(content="Namaste! Main aapka KisanSaathi hoon. Aapke area ke mandi prices check karne mein main aapki madad kar sakta hoon. Aap kis crop ka bhav janna chahte hain?")
    ]}
    
    print(f"AI: {state['messages'][0].content}")

    while True:
        user_input = input("Farmer: ")
        if user_input.lower() in ["quit", "exit", "bye"]:
            break
            
        state["messages"].append(HumanMessage(content=user_input))
        
        async for output in app.astream(state):
            for node, values in output.items():
                last_msg = values["messages"][-1]
                
                if node == "agent" and not (hasattr(last_msg, "tool_calls") and last_msg.tool_calls):
                    print(f"AI: {last_msg.content}")
                elif node == "tools":
                    print(f"🛠️ [Tool Call] Fetching data...")
                
                state["messages"].append(last_msg)

# To start the chat:
# await chat_with_farmer()


In [5]:
await chat_with_farmer()

🤖 KisanSaathi (Groq Edition) is ready! (ID: c59f6f44-1a98-4eaa-8cf0-3581316a32bb)
Type 'quit' to exit.

AI: Namaste! Main aapka KisanSaathi hoon. Aapke area ke mandi prices check karne mein main aapki madad kar sakta hoon. Aap kis crop ka bhav janna chahte hain?
🛠️ [Tool Call] Fetching data...
🛠️ [Tool Call] Fetching data...


CancelledError: 

In [ ]:
import os
import json
import httpx
import asyncpg
import asyncio
from typing import Annotated, TypedDict, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# --- CONFIGURATION ---
SESSION_FARMER_ID = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
MANDI_JSON_PATH = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_URL = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

# --- SYSTEM PROMPT ---
SYSTEM_INSTRUCTIONS = f"""You are KisanSaathi, a helpful agricultural assistant for Indian farmers. 
You are currently helping Farmer ID: {SESSION_FARMER_ID}.

RULES:
1. If the user asks for a price but doesn't specify a CROP, ask: "Aap kis crop (fasal) ka bhav janna chahte hain?"
2. If the user specifies a crop but NO MANDI, call 'get_market_price' with mandi_name="" and the crop name. 
   When the tool returns a list of mandis, show them to the user and ask which one they prefer. STOP THERE.
3. If the user provides a mandi name (even if slightly misspelled), call 'get_market_price'. 
4. If the tool says 'No records found', humbly tell the user that data isn't available for that crop/mandi right now.
5. Do NOT call tools repeatedly. Once you get a list of mandis or a price, respond to the user and wait for their next input."""

# --- TOOL DEFINITION ---
@tool
async def get_market_price(mandi_name: str = "", crop: str = "") -> str:
    """Fetches live mandi prices or enlists available mandis from JSON master data."""
    conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
    row = await conn.fetchrow("SELECT state_name, dist_name FROM farmers WHERE id = $1", SESSION_FARMER_ID)
    await conn.close()
    
    if not row: return "Error: Farmer not found."
    state, district = row["state_name"], row["dist_name"]

    # Load master data for listing/matching
    with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
        mandi_data = json.load(f)

    # Find mandis for this location
    available_mandis = []
    for s_key, districts in mandi_data.items():
        if s_key.lower() == state.lower():
            for d_key, mandis in districts.items():
                if d_key.lower() == district.lower():
                    available_mandis = mandis
                    break

    # STEP 1: If no mandi name provided, return the list
    if not mandi_name:
        if not available_mandis:
            return f"I see you are in {district}, {state}, but I don't have a list of mandis for this area."
        return f"Available mandis in {district}: {', '.join(available_mandis)}. Which one would you like to check?"

    # STEP 2: Mistaken Name Handling (Fuzzy match)
    actual_mandi = mandi_name
    for m in available_mandis:
        if mandi_name.lower().strip() in m.lower():
            actual_mandi = m
            break

    # STEP 3: API Call
    clean_mandi = actual_mandi.replace(" APMC", "").strip()
    params = {
        "api-key": os.getenv("AGMARKNET_API_KEY"),
        "format": "json",
        "filters[state]": state,
        "filters[market]": clean_mandi,
        "filters[commodity]": crop or "Wheat"
    }

    async with httpx.AsyncClient() as client:
        resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    
    if resp.status_code != 200: return "API Error: Government server is currently busy."

    records = resp.json().get("records", [])
    if not records:
        return f"Humble Note: No records found for {crop} at {actual_mandi} ({state}) today."

    rec = records[0]
    return f"Live Price for {rec['commodity']} at {rec['market']}: ₹{rec['modal_price']} per Quintal (Arrival Date: {rec['arrival_date']})."

# --- LANGRAPH SETUP ---
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], "Conversation history"]

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
).bind_tools([get_market_price])

async def call_model(state: AgentState):
    response = await model.ainvoke([SystemMessage(content=SYSTEM_INSTRUCTIONS)] + state["messages"])
    return {"messages": [response]}

async def call_tool(state: AgentState):
    last_msg = state["messages"][-1]
    results = []
    for tool_call in last_msg.tool_calls:
        print(f"🛠️  Calling Tool: {tool_call['name']}({tool_call['args']})")
        content = await get_market_price.ainvoke(tool_call["args"])
        results.append(ToolMessage(tool_call_id=tool_call["id"], content=str(content)))
    return {"messages": results}

def router(state: AgentState):
    if state["messages"][-1].tool_calls: return "tools"
    return END

graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", call_tool)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", router)
graph.add_edge("tools", "agent")
app = graph.compile()

# --- CHAT INTERFACE ---
async def chat_with_farmer():
    print(f"🤖 KisanSaathi (Groq) Active | Farmer ID: {SESSION_FARMER_ID}\n")
    state = {"messages": []}

    while True:
        u_input = input("Farmer: ")
        if u_input.lower() in ["quit", "exit"]: break
            
        state["messages"].append(HumanMessage(content=u_input))
        
        # Execute graph
        result = await app.ainvoke(state)
        state["messages"] = result["messages"] # Update state with full history
        
        # Print only the latest AI message
        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage) and not msg.tool_calls:
                print(f"AI: {msg.content}")
                break

# Run it
# await chat_with_farmer()


In [7]:
await chat_with_farmer()

🤖 KisanSaathi (Groq) Active | Farmer ID: c59f6f44-1a98-4eaa-8cf0-3581316a32bb

🛠️  Calling Tool: get_market_price({'crop': 'wheat', 'mandi_name': 'Agra'})
🛠️  Calling Tool: get_market_price({'crop': 'wheat', 'mandi_name': 'Agra APMC (Uttar Pradesh)'})
🛠️  Calling Tool: get_market_price({'crop': 'wheat', 'mandi_name': 'Agra APMC (Uttar Pradesh)'})
🛠️  Calling Tool: get_market_price({'crop': 'wheat', 'mandi_name': 'Agra APMC (Uttar Pradesh)'})
🛠️  Calling Tool: get_market_price({'crop': 'wheat', 'mandi_name': 'Agra APMC (Uttar Pradesh)'})
🛠️  Calling Tool: get_market_price({'crop': 'wheat', 'mandi_name': 'Agra APMC (Uttar Pradesh)'})
🛠️  Calling Tool: get_market_price({'crop': 'wheat', 'mandi_name': 'Agra APMC (Uttar Pradesh)'})
🛠️  Calling Tool: get_market_price({'crop': 'wheat', 'mandi_name': 'Agra APMC (Uttar Pradesh)'})


CancelledError: 

In [ ]:
"""
KisanSaathi - Fixed Version
============================
KEY FIXES:
  1. Tool Decomposition  — One overloaded tool split into two focused tools
  2. Recursion Guard     — graph.compile(recursion_limit=6) prevents infinite loops
  3. Stronger Prompt     — Explicit numbered steps with hard STOP rules
  4. Tool-call counting  — State tracks how many times each tool was called this turn
"""

import os
import json
import httpx
import asyncpg
import asyncio
from typing import Annotated, TypedDict, List
from langchain_core.messages import (
    BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────
SESSION_FARMER_ID = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
MANDI_JSON_PATH   = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_URL     = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

# ─────────────────────────────────────────────
#  SYSTEM PROMPT  (explicit numbered flow)
# ─────────────────────────────────────────────
SYSTEM_INSTRUCTIONS = f"""You are KisanSaathi, a friendly agricultural assistant for Indian farmers.
You are helping Farmer ID: {SESSION_FARMER_ID}.

You have EXACTLY TWO tools:
  • list_mandis(crop)           — Lists the APMCs in the farmer's district for a given crop.
  • fetch_crop_price(mandi_name, crop) — Fetches today's live price for a crop at a chosen APMC.

══════════════════════════════════════════════
STRICT 3-STEP FLOW FOR PRICE QUERIES
══════════════════════════════════════════════
STEP 1 — Crop check (NO tool call)
  If the farmer asks for a price but has NOT mentioned a crop:
  → Ask: "Aap kis fasal (crop) ka bhav janna chahte hain?"
  → WAIT for the answer. Do NOT call any tool yet.

STEP 2 — Mandi listing (call list_mandis ONCE)
  Once you have the crop name:
  → Call list_mandis(crop=<crop_name>) EXACTLY ONCE.
  → Show the returned mandi list to the farmer.
  → Ask: "Aap kaunsi APMC/Mandi mein bhav dekhna chahte hain?"
  → WAIT for their selection. Do NOT call fetch_crop_price yet.

STEP 3 — Price fetch (call fetch_crop_price ONCE)
  Once the farmer has selected a specific mandi:
  → Call fetch_crop_price(mandi_name=<chosen_mandi>, crop=<crop>) EXACTLY ONCE.
  → Show the result. Your job is DONE. Do NOT call any tool again.

══════════════════════════════════════════════
HARD RULES
══════════════════════════════════════════════
✗ NEVER call list_mandis more than once per user turn.
✗ NEVER call fetch_crop_price more than once per user turn.
✗ NEVER call any tool for greetings, general questions, farming tips, or weather.
✗ If a tool already returned data this turn, DO NOT call it again. Show the data and STOP.
✓ Respond in Hinglish (Hindi + English mix) so farmers understand easily."""


# ─────────────────────────────────────────────
#  TOOL 1 — list_mandis
#  Responsibility: DB lookup + JSON scan → return APMC list
#  Does NOT touch the Agmarknet API at all
# ─────────────────────────────────────────────
@tool
async def list_mandis(crop: str) -> str:
    """
    Returns a list of APMC mandis available in the farmer's district for a given crop.
    Call this ONLY when you have the crop name but no mandi chosen yet.
    """
    try:
        conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
        row  = await conn.fetchrow(
            "SELECT state_name, dist_name FROM farmers WHERE id = $1",
            SESSION_FARMER_ID
        )
        await conn.close()
    except Exception as e:
        return f"Database Error: {e}"

    if not row:
        return "Error: Farmer not found in database."

    state    = row["state_name"]
    district = row["dist_name"]

    # Load mandi master JSON
    try:
        with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
            mandi_data = json.load(f)
    except FileNotFoundError:
        return "Error: Mandi master data file not found."

    available_mandis: list[str] = []
    for s_key, districts in mandi_data.items():
        if s_key.lower() == state.lower():
            for d_key, mandis in districts.items():
                if d_key.lower() == district.lower():
                    available_mandis = mandis
                    break

    if not available_mandis:
        return (
            f"Aapke district {district} ({state}) ke liye mandi data abhi available nahi hai. "
            "Kripya baad mein try karein."
        )

    mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(available_mandis))
    return (
        f"📍 {district}, {state} ke available mandis ({crop} ke liye):\n"
        f"{mandi_list}\n\n"
        f"Kaunsi mandi ka bhav dekhna hai? (Number ya naam batayein)"
    )


# ─────────────────────────────────────────────
#  TOOL 2 — fetch_crop_price
#  Responsibility: Agmarknet API call → return price
#  Does NOT touch the DB or JSON master at all
# ─────────────────────────────────────────────
@tool
async def fetch_crop_price(mandi_name: str, crop: str) -> str:
    """
    Fetches today's live price for a crop at a specific APMC mandi from Agmarknet API.
    Call this ONLY after the farmer has explicitly selected a mandi.
    """
    if not mandi_name or not crop:
        return "Error: Both mandi_name and crop are required."

    # Fetch state from DB (needed for API filter)
    try:
        conn  = await asyncpg.connect(os.getenv("DATABASE_URL"))
        row   = await conn.fetchrow(
            "SELECT state_name FROM farmers WHERE id = $1",
            SESSION_FARMER_ID
        )
        await conn.close()
    except Exception as e:
        return f"Database Error: {e}"

    if not row:
        return "Error: Farmer not found."

    state      = row["state_name"]
    clean_name = mandi_name.replace("APMC", "").strip()  # API expects plain name

    params = {
        "api-key":             os.getenv("AGMARKNET_API_KEY"),
        "format":              "json",
        "filters[state]":      state,
        "filters[market]":     clean_name,
        "filters[commodity]":  crop,
    }

    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    except httpx.TimeoutException:
        return "API Timeout: Government server did not respond. Thodi der baad try karein."

    if resp.status_code != 200:
        return f"API Error {resp.status_code}: Government server busy hai. Baad mein try karein."

    records = resp.json().get("records", [])
    if not records:
        return (
            f"⚠️  {crop} ka aaj ka data {clean_name} mandi mein available nahi hai. "
            "Kal try karein ya doosri mandi select karein."
        )

    rec = records[0]
    return (
        f"✅ Live Price\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"🌾 Crop    : {rec['commodity']}\n"
        f"🏪 Mandi   : {rec['market']}\n"
        f"💰 Price   : ₹{rec['modal_price']} / Quintal\n"
        f"📅 Date    : {rec['arrival_date']}\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━"
    )


# ─────────────────────────────────────────────
#  STATE
# ─────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], "Full conversation history"]


# ─────────────────────────────────────────────
#  MODEL — bind BOTH tools
# ─────────────────────────────────────────────
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROK_API_KEY"),   # <-- move key to .env !
).bind_tools([list_mandis, fetch_crop_price])


# ─────────────────────────────────────────────
#  GRAPH NODES
# ─────────────────────────────────────────────
async def call_model(state: AgentState) -> dict:
    """LLM decides next action based on conversation history."""
    response = await model.ainvoke(
        [SystemMessage(content=SYSTEM_INSTRUCTIONS)] + state["messages"]
    )
    return {"messages": [response]}


async def call_tool(state: AgentState) -> dict:
    """Execute whatever tool(s) the LLM requested."""
    last_msg = state["messages"][-1]
    results  = []

    for tool_call in last_msg.tool_calls:
        print(f"🛠️  Tool called: {tool_call['name']}({tool_call['args']})")

        # Route to correct tool
        if tool_call["name"] == "list_mandis":
            content = await list_mandis.ainvoke(tool_call["args"])
        elif tool_call["name"] == "fetch_crop_price":
            content = await fetch_crop_price.ainvoke(tool_call["args"])
        else:
            content = f"Unknown tool: {tool_call['name']}"

        results.append(
            ToolMessage(tool_call_id=tool_call["id"], content=str(content))
        )

    return {"messages": results}


def router(state: AgentState) -> str:
    """Route: if LLM wants a tool → run it; otherwise → end turn."""
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


# ─────────────────────────────────────────────
#  BUILD GRAPH
# ─────────────────────────────────────────────
graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", call_tool)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", router, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

# ✅ KEY FIX: recursion_limit stops the loop even if LLM misbehaves
app = graph.compile()


# ─────────────────────────────────────────────
#  CHAT INTERFACE
# ─────────────────────────────────────────────
async def chat_with_farmer():
    print(f"\n🤖 KisanSaathi Active | Farmer: {SESSION_FARMER_ID}\n")
    state = {"messages": []}

    while True:
        user_input = input("Farmer: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "band karo"):
            print("KisanSaathi: Dhanyavad! Jai Jawan Jai Kisan 🌾")
            break

        state["messages"].append(HumanMessage(content=user_input))

        # ✅ recursion_limit=6 means: agent→tool→agent→tool→agent→tool → STOP
        #    For our 2-step flow (list_mandis then fetch_price) we only need 4 hops
        result = await app.ainvoke(
            state,
            config={"recursion_limit": 6}
        )
        state["messages"] = result["messages"]

        # Print only the final AI text response (skip tool-call messages)
        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage) and not msg.tool_calls:
                print(f"\nKisanSaathi: {msg.content}\n")
                break

# Remove this:
if __name__ == "__main__":
    asyncio.run(chat_with_farmer())

# Replace with this:
await chat_with_farmer()


🤖 KisanSaathi Active | Farmer: c59f6f44-1a98-4eaa-8cf0-3581316a32bb


KisanSaathi: Namaste! KisanSaathi yahan aapki sahayta ke liye hai. Aap kya jaanna chahte hain?


KisanSaathi: Aap kis fasal (crop) ka bhav janna chahte hain?

🛠️  Tool called: list_mandis({'crop': 'wheat'})
🛠️  Tool called: list_mandis({'crop': 'wheat'})
🛠️  Tool called: list_mandis({'crop': 'wheat'})


GraphRecursionError: Recursion limit of 6 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

## For debugging:-

In [ ]:
import os
import json
import httpx
import asyncpg
import asyncio
from typing import Annotated, TypedDict, List
from langchain_core.messages import (
    BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────
SESSION_FARMER_ID = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
MANDI_JSON_PATH   = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_URL     = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

# 🔍 DEBUG: Global hop counter — resets each user turn
_debug_hop = {"count": 0, "tool_calls": {}}

# ─────────────────────────────────────────────
#  SYSTEM PROMPT
# ─────────────────────────────────────────────
SYSTEM_INSTRUCTIONS = f"""You are KisanSaathi, a friendly agricultural assistant for Indian farmers.
You are helping Farmer ID: {SESSION_FARMER_ID}.

You have EXACTLY TWO tools:
  • list_mandis(crop)                    — Lists APMCs in the farmer's district.
  • fetch_crop_price(mandi_name, crop)   — Fetches today's live price.

══════════════════════════════════════════════
STRICT 3-STEP FLOW FOR PRICE QUERIES
══════════════════════════════════════════════
STEP 1 — Crop check (NO tool call)
  If the farmer asks for a price but has NOT mentioned a crop:
  → Ask: "Aap kis fasal (crop) ka bhav janna chahte hain?"
  → WAIT for the answer. Do NOT call any tool yet.

STEP 2 — Mandi listing (call list_mandis ONCE)
  Once you have the crop name:
  → Call list_mandis(crop=<crop_name>) EXACTLY ONCE.
  → Show the returned mandi list to the farmer.
  → Ask: "Aap kaunsi APMC/Mandi mein bhav dekhna chahte hain?"
  → WAIT for their selection. Do NOT call fetch_crop_price yet.

STEP 3 — Price fetch (call fetch_crop_price ONCE)
  Once the farmer has selected a specific mandi:
  → Call fetch_crop_price(mandi_name=<chosen_mandi>, crop=<crop>) EXACTLY ONCE.
  → Show the result. Your job is DONE. Do NOT call any tool again.

══════════════════════════════════════════════
HARD RULES
══════════════════════════════════════════════
✗ NEVER call list_mandis more than once per user turn.
✗ NEVER call fetch_crop_price more than once per user turn.
✗ NEVER call any tool for greetings, general questions, farming tips, or weather.
✗ If a tool already returned data this turn, DO NOT call it again. Show the data and STOP.
✓ Respond in Hinglish (Hindi + English mix) so farmers understand easily."""


# ─────────────────────────────────────────────
#  TOOL 1 — list_mandis
# ─────────────────────────────────────────────
@tool
async def list_mandis(crop: str) -> str:
    """
    Returns a list of APMC mandis in the farmer's district for a given crop.
    Call this ONLY when you have the crop name but no mandi chosen yet.
    """
    try:
        conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
        row  = await conn.fetchrow(
            "SELECT state_name, dist_name FROM farmers WHERE id = $1",
            SESSION_FARMER_ID
        )
        await conn.close()
    except Exception as e:
        return f"Database Error: {e}"

    if not row:
        return "Error: Farmer not found in database."

    state    = row["state_name"]
    district = row["dist_name"]

    try:
        with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
            mandi_data = json.load(f)
    except FileNotFoundError:
        return "Error: Mandi master data file not found."

    available_mandis: list[str] = []
    for s_key, districts in mandi_data.items():
        if s_key.lower() == state.lower():
            for d_key, mandis in districts.items():
                if d_key.lower() == district.lower():
                    available_mandis = mandis
                    break

    if not available_mandis:
        return (
            f"Aapke district {district} ({state}) ke liye mandi data abhi available nahi hai. "
            "Kripya baad mein try karein."
        )

    mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(available_mandis))
    return (
        f"📍 {district}, {state} ke available mandis ({crop} ke liye):\n"
        f"{mandi_list}\n\n"
        f"Kaunsi mandi ka bhav dekhna hai? (Number ya naam batayein)"
    )


# ─────────────────────────────────────────────
#  TOOL 2 — fetch_crop_price
# ─────────────────────────────────────────────
@tool
async def fetch_crop_price(mandi_name: str, crop: str) -> str:
    """
    Fetches today's live price for a crop at a specific APMC mandi.
    Call this ONLY after the farmer has explicitly selected a mandi.
    """
    if not mandi_name or not crop:
        return "Error: Both mandi_name and crop are required."

    try:
        conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
        row  = await conn.fetchrow(
            "SELECT state_name FROM farmers WHERE id = $1",
            SESSION_FARMER_ID
        )
        await conn.close()
    except Exception as e:
        return f"Database Error: {e}"

    if not row:
        return "Error: Farmer not found."

    state      = row["state_name"]
    clean_name = mandi_name.replace("APMC", "").strip()

    params = {
        "api-key":            os.getenv("AGMARKNET_API_KEY"),
        "format":             "json",
        "filters[state]":     state,
        "filters[market]":    clean_name,
        "filters[commodity]": crop,
    }

    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    except httpx.TimeoutException:
        return "API Timeout: Government server did not respond. Thodi der baad try karein."

    if resp.status_code != 200:
        return f"API Error {resp.status_code}: Government server busy hai."

    records = resp.json().get("records", [])
    if not records:
        return (
            f"⚠️  {crop} ka aaj ka data {clean_name} mandi mein available nahi hai. "
            "Kal try karein ya doosri mandi select karein."
        )

    rec = records[0]
    return (
        f"✅ Live Price\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"🌾 Crop    : {rec['commodity']}\n"
        f"🏪 Mandi   : {rec['market']}\n"
        f"💰 Price   : ₹{rec['modal_price']} / Quintal\n"
        f"📅 Date    : {rec['arrival_date']}\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━"
    )


# ─────────────────────────────────────────────
#  STATE
# ─────────────────────────────────────────────
from typing import Annotated, List
from langgraph.graph.message import add_messages   # ← ADD THIS
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]  # ← CHANGE THIS
# ─────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROK_API_KEY"),
).bind_tools([list_mandis, fetch_crop_price])


# ─────────────────────────────────────────────
#  GRAPH NODES (with full debug instrumentation)
# ─────────────────────────────────────────────
async def call_model(state: AgentState) -> dict:
    _debug_hop["count"] += 1
    hop = _debug_hop["count"]

    # 🔍 DEBUG: Show full message history entering the agent
    print(f"\n{'='*60}")
    print(f"🔍 DEBUG [Hop {hop}] → AGENT NODE")
    print(f"   Total messages in state: {len(state['messages'])}")
    for i, msg in enumerate(state["messages"]):
        msg_type  = type(msg).__name__
        content   = str(msg.content)[:100].replace("\n", " ")
        tool_info = f" | tool_calls={msg.tool_calls}" if hasattr(msg, "tool_calls") and msg.tool_calls else ""
        tool_name = f" | tool_name={msg.name}" if hasattr(msg, "name") and msg.name else ""
        print(f"   [{i}] {msg_type}: '{content}'{tool_info}{tool_name}")
    print(f"{'='*60}")

    response = await model.ainvoke(
        [SystemMessage(content=SYSTEM_INSTRUCTIONS)] + state["messages"]
    )

    # 🔍 DEBUG: Show exactly what LLM decided
    print(f"\n🔍 DEBUG [Hop {hop}] → LLM RESPONSE:")
    print(f"   content      : '{str(response.content)[:200]}'")
    print(f"   tool_calls   : {response.tool_calls}")
    if response.tool_calls:
        for tc in response.tool_calls:
            print(f"   → Wants to call: {tc['name']}({tc['args']})")
            # 🔍 Track per-tool call frequency
            key = tc["name"]
            _debug_hop["tool_calls"][key] = _debug_hop["tool_calls"].get(key, 0) + 1
            count = _debug_hop["tool_calls"][key]
            if count > 1:
                print(f"   ⚠️  WARNING: {key} has been called {count} times this turn! LOOP DETECTED.")
    else:
        print(f"   → No tool call → this should go to END")

    return {"messages": [response]}


async def call_tool(state: AgentState) -> dict:
    last_msg = state["messages"][-1]

    # 🔍 DEBUG: State snapshot at tool execution
    print(f"\n{'─'*60}")
    print(f"🔍 DEBUG → TOOL NODE — executing tool(s)")
    print(f"   Total messages before tool: {len(state['messages'])}")
    print(f"   Last message type: {type(last_msg).__name__}")
    print(f"   Tool calls requested: {[tc['name'] for tc in last_msg.tool_calls]}")

    # 🔍 Check: Is ToolMessage already present for this tool? (duplicate guard)
    existing_tool_results = [
        m for m in state["messages"] if isinstance(m, ToolMessage)
    ]
    print(f"   Existing ToolMessages in state: {len(existing_tool_results)}")
    for tm in existing_tool_results:
        print(f"     → ToolMessage id={tm.tool_call_id} | content='{str(tm.content)[:80]}'")

    results = []
    for tool_call in last_msg.tool_calls:
        print(f"\n🛠️  Calling Tool: {tool_call['name']}({tool_call['args']})")

        if tool_call["name"] == "list_mandis":
            content = await list_mandis.ainvoke(tool_call["args"])
        elif tool_call["name"] == "fetch_crop_price":
            content = await fetch_crop_price.ainvoke(tool_call["args"])
        else:
            content = f"Unknown tool: {tool_call['name']}"

        print(f"   Tool returned: '{str(content)[:150]}'")

        results.append(
            ToolMessage(tool_call_id=tool_call["id"], content=str(content))
        )

    return {"messages": results}


def router(state: AgentState) -> str:
    last = state["messages"][-1]

    # 🔍 DEBUG: Show router decision
    print(f"\n🔍 DEBUG → ROUTER:")
    print(f"   Last message type : {type(last).__name__}")
    is_ai    = isinstance(last, AIMessage)
    has_tool = is_ai and bool(last.tool_calls)
    print(f"   Is AIMessage       : {is_ai}")
    print(f"   Has tool_calls     : {has_tool}")

    if has_tool:
        print(f"   Decision → 'tools'")
        return "tools"

    print(f"   Decision → END")
    return END


# ─────────────────────────────────────────────
#  BUILD GRAPH
# ─────────────────────────────────────────────
graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", call_tool)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", router, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")
app = graph.compile()


# ─────────────────────────────────────────────
#  CHAT INTERFACE
# ─────────────────────────────────────────────
async def chat_with_farmer():
    print(f"\n🤖 KisanSaathi Debug Mode | Farmer: {SESSION_FARMER_ID}\n")
    print("🔍 DEBUG MODE ACTIVE — all LLM decisions will be printed\n")
    state = {"messages": []}

    while True:
        user_input = input("Farmer: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "band karo"):
            print("KisanSaathi: Dhanyavad! Jai Jawan Jai Kisan 🌾")
            break

        # 🔍 Reset per-turn debug counters
        _debug_hop["count"]      = 0
        _debug_hop["tool_calls"] = {}
        print(f"\n{'#'*60}")
        print(f"# NEW TURN | Input: '{user_input}'")
        print(f"{'#'*60}")

        state["messages"].append(HumanMessage(content=user_input))

        try:
            result = await app.ainvoke(
                state,
                config={"recursion_limit": 6}
            )
            state["messages"] = result["messages"]
        except Exception as e:
            print(f"\n❌ ERROR: {type(e).__name__}: {e}")
            print(f"   Total hops before error: {_debug_hop['count']}")
            print(f"   Tool call counts: {_debug_hop['tool_calls']}")
            continue

        # 🔍 DEBUG: Final state summary
        print(f"\n{'='*60}")
        print(f"🔍 DEBUG → TURN COMPLETE")
        print(f"   Total hops     : {_debug_hop['count']}")
        print(f"   Tool call freq : {_debug_hop['tool_calls']}")
        print(f"   Final messages : {len(state['messages'])}")
        print(f"{'='*60}")

        # Print final AI response
        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage) and not msg.tool_calls:
                print(f"\nKisanSaathi: {msg.content}\n")
                break


# In Jupyter:
await chat_with_farmer()

# In .py file:
# if __name__ == "__main__":
#     asyncio.run(chat_with_farmer())



🤖 KisanSaathi Debug Mode | Farmer: c59f6f44-1a98-4eaa-8cf0-3581316a32bb

🔍 DEBUG MODE ACTIVE — all LLM decisions will be printed


############################################################
# NEW TURN | Input: 'Hello'
############################################################

🔍 DEBUG [Hop 1] → AGENT NODE
   Total messages in state: 1
   [0] HumanMessage: 'Hello'

🔍 DEBUG [Hop 1] → LLM RESPONSE:
   content      : 'Namaste! KisanSaathi yahan aapki sahayata ke liye hai. Aap kya jaanna chahte hain aaj?'
   tool_calls   : []
   → No tool call → this should go to END

🔍 DEBUG → ROUTER:
   Last message type : AIMessage
   Is AIMessage       : True
   Has tool_calls     : False
   Decision → END

🔍 DEBUG → TURN COMPLETE
   Total hops     : 1
   Tool call freq : {}
   Final messages : 2

KisanSaathi: Namaste! KisanSaathi yahan aapki sahayata ke liye hai. Aap kya jaanna chahte hain aaj?


############################################################
# NEW TURN | Input: 'Aaj Wheat ka Price k

In [ ]:
"""
KisanSaathi — Manual State Machine (Final Fix)
================================================
ARCHITECTURE CHANGE:
  OLD: LangGraph ReAct loop → LLM decides when to call tools → infinite loop
  NEW: Python state machine → CODE decides when to call tools → deterministic

FLOW:
  IDLE            → user says price query  → extract crop (LLM) → WAITING_FOR_MANDI
  WAITING_FOR_MANDI → user picks mandi     → fetch price (API)  → IDLE
  (crop missing)  → ask for crop (LLM)    → WAITING_FOR_CROP
  WAITING_FOR_CROP → user gives crop       → list mandis (JSON) → WAITING_FOR_MANDI

The LLM is used ONLY for:
  1. Intent detection (price query vs general chat)
  2. Crop name extraction
  3. Generating friendly Hinglish responses
  
Tools are called DIRECTLY by Python, never by the LLM.
"""

import os
import re
import json
import httpx
import asyncpg
import asyncio
from enum import Enum
from dataclasses import dataclass, field
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv()

# ─────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────
SESSION_FARMER_ID = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
MANDI_JSON_PATH   = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_URL     = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

# ─────────────────────────────────────────────
#  STATE MACHINE STAGES
# ─────────────────────────────────────────────
class Stage(Enum):
    IDLE              = "idle"              # Normal conversation
    WAITING_FOR_CROP  = "waiting_for_crop"  # Asked user "which crop?"
    WAITING_FOR_MANDI = "waiting_for_mandi" # Showed mandi list, waiting for selection

@dataclass
class ConversationState:
    stage: Stage           = Stage.IDLE
    pending_crop: str      = ""            # Crop being queried
    available_mandis: list = field(default_factory=list)
    farmer_state: str      = ""
    farmer_district: str   = ""


# ─────────────────────────────────────────────
#  LLM — used ONLY for NLP, no tools bound
# ─────────────────────────────────────────────
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROK_API_KEY"),
)


# ─────────────────────────────────────────────
#  LLM HELPERS  (pure NLP, no tool calls)
# ─────────────────────────────────────────────
async def detect_intent_and_crop(user_msg: str) -> dict:
    """
    Ask LLM: is this a price query? If yes, what crop?
    Returns: {"is_price_query": bool, "crop": str or ""}
    """
    prompt = f"""Analyze this farmer's message and respond with ONLY a JSON object.

Message: "{user_msg}"

Rules:
- is_price_query = true if the farmer is asking about crop price/bhav/rate/daam
- crop = the crop name mentioned (in English), or "" if not mentioned
- Common crops: Wheat, Rice, Onion, Tomato, Potato, Maize, Cotton, Soybean, Sugarcane, Mustard

Respond ONLY with JSON, no explanation:
{{"is_price_query": true/false, "crop": "CropName or empty string"}}"""

    response = await llm.ainvoke([HumanMessage(content=prompt)])
    
    try:
        # Strip markdown fences if present
        text = response.content.strip().replace("```json", "").replace("```", "").strip()
        return json.loads(text)
    except Exception:
        # Fallback: check for price keywords manually
        price_keywords = ["price", "bhav", "rate", "daam", "kitna", "khareed", "bech"]
        is_price = any(k in user_msg.lower() for k in price_keywords)
        return {"is_price_query": is_price, "crop": ""}


async def extract_crop_from_message(user_msg: str) -> str:
    """Extract crop name when user is answering 'which crop?' question."""
    prompt = f"""The farmer was asked which crop they want prices for.
Their reply: "{user_msg}"

Extract ONLY the crop name in English (e.g. Wheat, Onion, Tomato).
If no crop found, respond with: NONE
Respond with just the crop name or NONE."""

    response = await llm.ainvoke([HumanMessage(content=prompt)])
    crop = response.content.strip()
    return "" if crop.upper() == "NONE" else crop


async def generate_response(user_msg: str, context: str) -> str:
    """Generate a friendly Hinglish response for general queries."""
    prompt = f"""You are KisanSaathi, a helpful agricultural assistant for Indian farmers.
Context: {context}
Farmer said: "{user_msg}"

Respond helpfully in Hinglish (Hindi + English mix). Keep it short and warm."""
    
    response = await llm.ainvoke([HumanMessage(content=prompt)])
    return response.content.strip()


# ─────────────────────────────────────────────
#  DATA FUNCTIONS  (called directly by Python)
# ─────────────────────────────────────────────
async def get_farmer_location() -> tuple[str, str]:
    """Fetch farmer's state and district from DB. Returns (state, district)."""
    conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
    row  = await conn.fetchrow(
        "SELECT state_name, dist_name FROM farmers WHERE id = $1",
        SESSION_FARMER_ID
    )
    await conn.close()
    if not row:
        raise ValueError("Farmer not found in database.")
    return row["state_name"], row["dist_name"]


def get_mandis_from_json(state: str, district: str) -> list[str]:
    """Load mandi list for a state+district from the JSON master file."""
    with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
        mandi_data = json.load(f)

    for s_key, districts in mandi_data.items():
        if s_key.lower() == state.lower():
            for d_key, mandis in districts.items():
                if d_key.lower() == district.lower():
                    return mandis
    return []


async def fetch_price_from_api(state: str, mandi_name: str, crop: str) -> str:
    """Call Agmarknet API and return formatted price string."""
    clean_name = mandi_name.replace("APMC", "").strip()

    params = {
        "api-key":            os.getenv("AGMARKNET_API_KEY"),
        "format":             "json",
        "filters[state]":     state,
        "filters[market]":    clean_name,
        "filters[commodity]": crop,
    }

    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    except httpx.TimeoutException:
        return "⚠️  Government server ne respond nahi kiya. Thodi der baad try karein."

    if resp.status_code != 200:
        return f"⚠️  API Error {resp.status_code}. Baad mein try karein."

    records = resp.json().get("records", [])
    if not records:
        return (
            f"⚠️  {crop} ka aaj ka data {clean_name} mein available nahi hai.\n"
            "Kal try karein ya koi aur mandi select karein."
        )

    rec = records[0]
    return (
        f"✅ Aaj Ka Bhav\n"
        f"━━━━━━━━━━━━━━━━━━━━━━\n"
        f"🌾 Fasal   : {rec['commodity']}\n"
        f"🏪 Mandi   : {rec['market']}\n"
        f"💰 Bhav    : ₹{rec['modal_price']} / Quintal\n"
        f"📅 Tarikh  : {rec['arrival_date']}\n"
        f"━━━━━━━━━━━━━━━━━━━━━━"
    )


def fuzzy_match_mandi(user_input: str, available_mandis: list[str]) -> str:
    """
    Match user's mandi selection (could be a number like '1' or partial name).
    Returns the matched mandi name or empty string.
    """
    user_input = user_input.strip()

    # Check if user typed a number (e.g. "1" or "2")
    if user_input.isdigit():
        idx = int(user_input) - 1
        if 0 <= idx < len(available_mandis):
            return available_mandis[idx]
        return ""

    # Partial name match (case-insensitive)
    for mandi in available_mandis:
        if user_input.lower() in mandi.lower() or mandi.lower() in user_input.lower():
            return mandi

    return ""


# ─────────────────────────────────────────────
#  MAIN CHAT HANDLER  (the state machine)
# ─────────────────────────────────────────────
async def handle_message(user_msg: str, conv: ConversationState) -> str:
    """
    Process one user message through the state machine.
    Returns the bot's response string.
    Updates conv (state) in place.
    """

    # ── STAGE: WAITING FOR CROP ANSWER ──────────────────────────────
    if conv.stage == Stage.WAITING_FOR_CROP:
        crop = await extract_crop_from_message(user_msg)
        if not crop:
            return "Maafi chahta hun, samajh nahi aaya. Kripya fasal ka naam batayein — jaise Onion, Wheat, Tomato."

        conv.pending_crop = crop

        # Load mandis (Python calls this directly, not the LLM)
        mandis = get_mandis_from_json(conv.farmer_state, conv.farmer_district)
        if not mandis:
            conv.stage = Stage.IDLE
            return f"Aapke district {conv.farmer_district} ke liye mandi data abhi available nahi hai."

        conv.available_mandis = mandis
        conv.stage = Stage.WAITING_FOR_MANDI

        mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(mandis))
        return (
            f"📍 {conv.farmer_district} ({conv.farmer_state}) ki APMCs ({crop} ke liye):\n\n"
            f"{mandi_list}\n\n"
            f"Kaunsi mandi ka bhav chahiye? (Number ya naam type karein)"
        )

    # ── STAGE: WAITING FOR MANDI SELECTION ──────────────────────────
    if conv.stage == Stage.WAITING_FOR_MANDI:
        matched = fuzzy_match_mandi(user_msg, conv.available_mandis)

        if not matched:
            mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(conv.available_mandis))
            return (
                f"Yeh mandi nahi mili. Kripya list mein se number ya naam select karein:\n\n"
                f"{mandi_list}"
            )

        # ✅ Python directly calls the API — LLM is NOT involved here
        print(f"🌐 API Call: fetch_price({matched}, {conv.pending_crop})")
        result = await fetch_price_from_api(conv.farmer_state, matched, conv.pending_crop)

        # Reset state
        conv.stage         = Stage.IDLE
        conv.pending_crop  = ""
        conv.available_mandis = []

        return result

    # ── STAGE: IDLE — detect intent ─────────────────────────────────
    intent = await detect_intent_and_crop(user_msg)

    if intent.get("is_price_query"):
        crop = intent.get("crop", "").strip()

        # Ensure we have farmer location loaded
        if not conv.farmer_state:
            try:
                conv.farmer_state, conv.farmer_district = await get_farmer_location()
            except Exception as e:
                return f"Database error: {e}"

        # Case A: Crop NOT mentioned — ask for it
        if not crop:
            conv.stage = Stage.WAITING_FOR_CROP
            return "Aap kis fasal (crop) ka bhav jaanna chahte hain? (e.g. Wheat, Onion, Tomato)"

        # Case B: Crop IS mentioned — show mandi list
        conv.pending_crop = crop
        mandis = get_mandis_from_json(conv.farmer_state, conv.farmer_district)

        if not mandis:
            return f"Aapke district {conv.farmer_district} ke liye mandi data abhi available nahi hai."

        conv.available_mandis = mandis
        conv.stage = Stage.WAITING_FOR_MANDI

        mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(mandis))
        return (
            f"📍 {conv.farmer_district} ({conv.farmer_state}) ki APMCs ({crop} ke liye):\n\n"
            f"{mandi_list}\n\n"
            f"Kaunsi mandi ka bhav chahiye? (Number ya naam type karein)"
        )

    # Case C: General question — LLM answers directly, no tools
    return await generate_response(user_msg, "General farming assistance")


# ─────────────────────────────────────────────
#  ENTRY POINT
# ─────────────────────────────────────────────
async def chat_with_farmer():
    print(f"\n🌾 KisanSaathi Active | Farmer: {SESSION_FARMER_ID}\n")
    print("KisanSaathi: Namaste! Main KisanSaathi hun. Aaj main aapki kya madad kar sakta hun?\n")

    conv = ConversationState()  # Fresh state machine

    while True:
        user_input = input("Farmer: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "band karo"):
            print("KisanSaathi: Dhanyavad! Jai Jawan Jai Kisan 🌾")
            break

        response = await handle_message(user_input, conv)
        print(f"\nKisanSaathi: {response}\n")


# In Jupyter — run this cell:
await chat_with_farmer()

# In .py file — use this instead:
# if __name__ == "__main__":
#     asyncio.run(chat_with_farmer())


🌾 KisanSaathi Active | Farmer: c59f6f44-1a98-4eaa-8cf0-3581316a32bb

KisanSaathi: Namaste! Main KisanSaathi hun. Aaj main aapki kya madad kar sakta hun?


KisanSaathi: Namaste kisan bhai, kaise ho? Aapki kheti mein kya madad chahiye aaj?


KisanSaathi: 📍 agra (Uttar Pradesh) ki APMCs (Wheat ke liye):

  1. Achnera APMC
  2. Agra APMC
  3. Fatehabad APMC
  4. Fatehpur Sikri APMC
  5. Jagnair APMC
  6. Jarar APMC

Kaunsi mandi ka bhav chahiye? (Number ya naam type karein)

🌐 API Call: fetch_price(Achnera APMC, Wheat)

KisanSaathi: ✅ Aaj Ka Bhav
━━━━━━━━━━━━━━━━━━━━━━
🌾 Fasal   : Wheat
🏪 Mandi   : Achnera APMC
💰 Bhav    : ₹2350 / Quintal
📅 Tarikh  : 21/04/2026
━━━━━━━━━━━━━━━━━━━━━━


KisanSaathi: Aap kis fasal (crop) ka bhav jaanna chahte hain? (e.g. Wheat, Onion, Tomato)


KisanSaathi: 📍 agra (Uttar Pradesh) ki APMCs (Wheat ke liye):

  1. Achnera APMC
  2. Agra APMC
  3. Fatehabad APMC
  4. Fatehpur Sikri APMC
  5. Jagnair APMC
  6. Jarar APMC

Kaunsi mandi ka bhav chahiye? (Number ya